1. Loop and Track Network Latency (Jitter)
 
 This script sends multiple requests to track how network latency bounces around over time. The variance between these spikes is called jitter

In [1]:
import time
import requests

url = "https://github.com"
iterations = 5
latency_history = []

print(f"Tracking latency to {url} over {iterations} requests:\n")

for i in range(1, iterations + 1):
    try:
        response = requests.get(url, timeout=5)
        # Convert response duration to milliseconds
        latency_ms = response.elapsed.total_seconds() * 1000
        latency_history.append(latency_ms)
        
        # Create a simple visual text bar to show scale
        visual_bar = "█" * int(latency_ms / 20)
        print(f"Req {i}: {latency_ms:6.1f} ms | {visual_bar}")
        
    except requests.exceptions.RequestException as e:
        print(f"Req {i}: Failed ({e})")
    
    # Pause briefly between requests
    time.sleep(0.5)

# Calculate summary metrics
if latency_history:
    avg_latency = sum(latency_history) / len(latency_history)
    jitter = max(latency_history) - min(latency_history)
    print(f"\n--- Summary Metrics ---")
    print(f"Average Latency: {avg_latency:.1f} ms")
    print(f"Jitter (Max - Min Variance): {jitter:.1f} ms")


Tracking latency to https://github.com over 5 requests:

Req 1:   90.1 ms | ████
Req 2:   85.5 ms | ████
Req 3:   93.8 ms | ████
Req 4:   87.1 ms | ████
Req 5:  121.6 ms | ██████

--- Summary Metrics ---
Average Latency: 95.6 ms
Jitter (Max - Min Variance): 36.1 ms


2. Measure Individual Network Steps

To see exactly why a request takes time, you can break down the lifecycle into components like DNS Lookup, TCP Connection, and Server Processing Time. We use a lower-level library called urllib3 (which runs under the hood of requests) to access these raw performance numbers

In [2]:
import socket
import time
import http.client
from urllib.parse import urlparse

def measure_network_steps(url):
    parsed_url = urlparse(url)
    host = parsed_url.netloc if parsed_url.netloc else parsed_url.path
    path = parsed_url.path if parsed_url.netloc else "/"
    port = 443 if parsed_url.scheme == "https" else 80

    print(f"Analyzing connection steps to: {host}...\n")

    # Step 1: DNS Lookup Time
    t0 = time.perf_counter()
    try:
        ip_address = socket.gethostbyname(host)
        t1 = time.perf_counter()
        dns_time = (t1 - t0) * 1000
    except socket.gaierror:
        print("❌ DNS Lookup failed. Please check the URL.")
        return

    # Step 2: TCP / SSL Handshake Time
    try:
        if port == 443:
            import ssl
            context = ssl.create_default_context()
            conn = http.client.HTTPSConnection(host, port, context=context, timeout=5)
        else:
            conn = http.client.HTTPConnection(host, port, timeout=5)
        
        t2 = time.perf_counter()
        conn.connect()
        t3 = time.perf_counter()
        connect_time = (t3 - t2) * 1000
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return

    # Step 3: Server Processing Time (Time to First Byte)
    t4 = time.perf_counter()
    conn.request("GET", path)
    response = conn.getresponse()
    _ = response.read(1) # Read just the first byte to measure response start
    t5 = time.perf_counter()
    server_time = (t5 - t4) * 1000

    conn.close()

    # Output Results
    print(f"📍 Target IP: {ip_address}")
    print(f"⏳ 1. DNS Lookup Time:      {dns_time:.2f} ms")
    print(f"🤝 2. TCP/TLS Handshake:     {connect_time:.2f} ms")
    print(f"⚡ 3. Server Response Time:  {server_time:.2f} ms")
    print(f"----------------------------------------")
    print(f"📦 Total Network Latency:    {dns_time + connect_time + server_time:.2f} ms")

if __name__ == "__main__":
    measure_network_steps("https://google.com")


Analyzing connection steps to: google.com...

📍 Target IP: 192.178.158.102
⏳ 1. DNS Lookup Time:      0.96 ms
🤝 2. TCP/TLS Handshake:     58.85 ms
⚡ 3. Server Response Time:  52.70 ms
----------------------------------------
📦 Total Network Latency:    112.50 ms


3. Interactive Custom URL Latency Tool (CLI)

This script turns your code into a re-usable command-line interface (CLI). You can pass it any URL and specify how many times to test it right from your terminal.

In [3]:
import sys
import time
import requests

def run_latency_tool():
    # Prompt user for input if not supplied via command line
    url = input("Enter target URL (e.g., https://google.com): ").strip()
    if not url.startswith(('http://', 'https://')):
        url = 'https://' + url

    try:
        count = int(input("Enter number of checks (default 5): ") or 5)
    except ValueError:
        count = 5

    print(f"\n🚀 Profiling {url} ({count} passes)...")
    
    for i in range(1, count + 1):
        try:
            start = time.perf_counter()
            response = requests.get(url, timeout=5)
            end = time.perf_counter()
            
            # Overall script round-trip time vs internal HTTP processing time
            total_ms = (end - start) * 1000
            http_ms = response.elapsed.total_seconds() * 1000
            
            print(f"Pass {i}: HTTP Time = {http_ms:.1f}ms | Total Processing = {total_ms:.1f}ms (Status: {response.status_code})")
        except requests.exceptions.RequestException as e:
            print(f"Pass {i}: ❌ Connection Failed: {type(e).__name__}")
        time.sleep(0.5)

if __name__ == "__main__":
    run_latency_tool()



🚀 Profiling https://google.com (10 passes)...
Pass 1: HTTP Time = 131.1ms | Total Processing = 271.2ms (Status: 200)
Pass 2: HTTP Time = 149.3ms | Total Processing = 298.6ms (Status: 200)
Pass 3: HTTP Time = 133.7ms | Total Processing = 284.3ms (Status: 200)
Pass 4: HTTP Time = 133.0ms | Total Processing = 263.6ms (Status: 200)
Pass 5: HTTP Time = 134.8ms | Total Processing = 262.9ms (Status: 200)
Pass 6: HTTP Time = 135.3ms | Total Processing = 269.3ms (Status: 200)
Pass 7: HTTP Time = 132.3ms | Total Processing = 259.9ms (Status: 200)
Pass 8: HTTP Time = 136.3ms | Total Processing = 277.4ms (Status: 200)
Pass 9: HTTP Time = 143.9ms | Total Processing = 269.2ms (Status: 200)
Pass 10: HTTP Time = 135.6ms | Total Processing = 277.3ms (Status: 200)


4. Wi-Fi Router vs. Internet Isolation Script

If your internet feels slow, this script isolates the exact source of the lag. It pings your local Wi-Fi router (local gateway) and a public internet server simultaneously to show you if your router is dropping packets or adding delay.


In [4]:
import os
import platform
import subprocess

def check_local_vs_remote():
    # Identify default local router IP (usually 192.168.1.1 or 192.168.0.1)
    # Using 192.168.1.1 as a standard domestic baseline fallback
    router_ip = "192.168.1.1" 
    dns_ip = "8.8.8.8"  # Google Public DNS
    
    # Adjust ping parameters based on operating system
    param = "-n" if platform.system().lower() == "windows" else "-c"
    
    print("Checking local Wi-Fi connection vs external internet...")
    
    # 1. Ping Router
    print(f"\n[1/2] Pinging local Wi-Fi router ({router_ip})...")
    router_cmd = ["ping", param, "3", router_ip]
    router_output = subprocess.run(router_cmd, stdout=subprocess.PIPE, text=True)
    print(router_output.stdout)
    
    # 2. Ping Internet
    print(f"[2/2] Pinging public internet ({dns_ip})...")
    dns_cmd = ["ping", param, "3", dns_ip]
    dns_output = subprocess.run(dns_cmd, stdout=subprocess.PIPE, text=True)
    print(dns_output.stdout)
    
    print("\n--- Diagnostic Guide ---")
    print("• Router numbers high (>50ms) or failing? -> Your local Wi-Fi signal is weak.")
    print("• Router numbers low (<5ms) but Internet high? -> Your ISP line is bottlenecked.")

if __name__ == "__main__":
    check_local_vs_remote()


Checking local Wi-Fi connection vs external internet...

[1/2] Pinging local Wi-Fi router (192.168.1.1)...
PING 192.168.1.1 (192.168.1.1): 56 data bytes
Request timeout for icmp_seq 0
Request timeout for icmp_seq 1

--- 192.168.1.1 ping statistics ---
3 packets transmitted, 0 packets received, 100.0% packet loss

[2/2] Pinging public internet (8.8.8.8)...
PING 8.8.8.8 (8.8.8.8): 56 data bytes
64 bytes from 8.8.8.8: icmp_seq=0 ttl=112 time=16.741 ms
64 bytes from 8.8.8.8: icmp_seq=1 ttl=112 time=12.739 ms
64 bytes from 8.8.8.8: icmp_seq=2 ttl=112 time=15.208 ms

--- 8.8.8.8 ping statistics ---
3 packets transmitted, 3 packets received, 0.0% packet loss
round-trip min/avg/max/stddev = 12.739/14.896/16.741/1.649 ms


--- Diagnostic Guide ---
• Router numbers high (>50ms) or failing? -> Your local Wi-Fi signal is weak.
• Router numbers low (<5ms) but Internet high? -> Your ISP line is bottlenecked.
